In [8]:
# ── BLOCK 1: Imports & Data Loading ──
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv(r"E:\NCI\Sem3\Thesis\implementation\data\creditcard_with_tiers.csv")

print("✅ Libraries loaded!")
print(f"Dataset shape: {df.shape}")

✅ Libraries loaded!
Dataset shape: (284807, 32)


In [9]:
# ── BLOCK 2: Prepare Data ──

# Separate features, target and risk tiers
X = df.drop(columns=['Class', 'Risk_Tier'])
y = df['Class']
risk_tiers = df['Risk_Tier']

# Scale Amount and Time
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])
X['Time'] = scaler.fit_transform(X[['Time']])

# Train/test split
X_train, X_test, y_train, y_test, rt_train, rt_test = train_test_split(
    X, y, risk_tiers, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Convert to PyTorch tensors
# Think of tensors as PyTorch's version of numpy arrays
X_train_tensor = torch.FloatTensor(X_train_bal.values)
y_train_tensor = torch.LongTensor(y_train_bal.values)
X_test_tensor  = torch.FloatTensor(X_test.values)
y_test_tensor  = torch.LongTensor(y_test.values)

# Create DataLoaders
# DataLoader feeds data to the model in small batches during training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

print("✅ Data prepared for PyTorch!")
print(f"Training samples: {len(X_train_tensor):,}")
print(f"Test samples: {len(X_test_tensor):,}")
print(f"Input features: {X_train_tensor.shape[1]}")

✅ Data prepared for PyTorch!
Training samples: 454,902
Test samples: 56,962
Input features: 30


In [10]:
# ── BLOCK 3: Build the L2D Neural Network ──

class L2DNetwork(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super(L2DNetwork, self).__init__()
        
        # SHARED LAYERS
        # Both heads share this base — like a common brain
        # that processes the transaction first
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        # HEAD 1 — CLASSIFIER HEAD
        # Answers: "Is this fraud or not?"
        self.classifier_head = nn.Linear(32, num_classes)
        
        # HEAD 2 — DEFERRAL HEAD  
        # Answers: "Should I defer this to a human?"
        self.deferral_head = nn.Linear(32, 1)
    
    def forward(self, x):
        # Step 1: Pass through shared layers
        shared_out = self.shared(x)
        
        # Step 2: Get classification scores
        class_logits = self.classifier_head(shared_out)
        
        # Step 3: Get deferral score
        defer_logit = self.deferral_head(shared_out)
        defer_prob = torch.sigmoid(defer_logit)
        
        return class_logits, defer_prob

# Create the model
input_dim = X_train_tensor.shape[1]  # 30 features
model = L2DNetwork(input_dim=input_dim)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print("✅ L2D Network built!")
print(f"Input dimensions: {input_dim}")
print(f"Total parameters: {total_params:,}")
print()
print("Network Architecture:")
print(model)

✅ L2D Network built!
Input dimensions: 30
Total parameters: 4,163

Network Architecture:
L2DNetwork(
  (shared): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
  )
  (classifier_head): Linear(in_features=32, out_features=2, bias=True)
  (deferral_head): Linear(in_features=32, out_features=1, bias=True)
)


In [11]:
# ── BLOCK 4: L2D Loss Function ──

def l2d_loss(class_logits, defer_prob, targets, cost=0.5):
    """
    L2D Loss Function — Mozannar & Sontag (2020)
    
    In plain English:
    The model is penalised for two things:
    1. Making wrong classification decisions autonomously
    2. Deferring too much (deferral has a cost too)
    
    cost = how expensive it is to defer to a human
    0.5 means deferral costs the same as being wrong
    """
    batch_size = targets.shape[0]
    
    # Classification loss — how wrong are the autonomous decisions?
    ce_loss = nn.CrossEntropyLoss(reduction='none')
    class_loss = ce_loss(class_logits, targets)
    
    # Deferral probability for each sample
    defer_p = defer_prob.squeeze()
    
    # Combined L2D loss:
    # If defer_p is high → model defers → pays cost c
    # If defer_p is low → model decides → pays classification error
    autonomous_loss = (1 - defer_p) * class_loss
    deferral_cost   = defer_p * cost
    
    total_loss = (autonomous_loss + deferral_cost).mean()
    
    return total_loss

print("✅ L2D Loss Function defined!")
print()
print("=== LOSS FUNCTION EXPLAINED ===")
print("For each transaction the model faces a trade-off:")
print("  Option A: Decide autonomously → penalised if WRONG")
print("  Option B: Defer to human → always pays a small cost")
print()
print("The model learns to defer when uncertainty > deferral cost")
print("This is exactly what makes L2D different from a baseline!")

✅ L2D Loss Function defined!

=== LOSS FUNCTION EXPLAINED ===
For each transaction the model faces a trade-off:
  Option A: Decide autonomously → penalised if WRONG
  Option B: Defer to human → always pays a small cost

The model learns to defer when uncertainty > deferral cost
This is exactly what makes L2D different from a baseline!


In [13]:
# ── BLOCK 5: Train the L2D Model ──

# Setup
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

# Track loss over epochs
train_losses = []

print("Starting L2D Training...")
print("=" * 45)

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    num_batches = 0
    
    for X_batch, y_batch in train_loader:
        # Step 1: Reset gradients
        optimizer.zero_grad()
        
        # Step 2: Forward pass — get predictions
        class_logits, defer_prob = model(X_batch)
        
        # Step 3: Calculate L2D loss
        loss = l2d_loss(class_logits, defer_prob, y_batch)
        
        # Step 4: Backward pass — learn from mistakes
        loss.backward()
        
        # Step 5: Update model weights
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg_loss:.4f}")

print("=" * 45)
print("✅ Training complete!")

Starting L2D Training...
Epoch  1/10 | Loss: 0.0037
Epoch  2/10 | Loss: 0.0038
Epoch  3/10 | Loss: 0.0038
Epoch  4/10 | Loss: 0.0031
Epoch  5/10 | Loss: 0.0035
Epoch  6/10 | Loss: 0.0032
Epoch  7/10 | Loss: 0.0033
Epoch  8/10 | Loss: 0.0030
Epoch  9/10 | Loss: 0.0030
Epoch 10/10 | Loss: 0.0026
✅ Training complete!


In [14]:
# ── BLOCK 6: Evaluate Standard L2D ──

model.eval()
with torch.no_grad():
    class_logits, defer_prob = model(X_test_tensor)
    
    # Get predictions
    class_pred = torch.argmax(class_logits, dim=1).numpy()
    defer_p    = defer_prob.squeeze().numpy()
    
    # A transaction is deferred if defer probability > 0.5
    deferred   = defer_p > 0.5
    autonomous = ~deferred

print("=== STANDARD L2D RESULTS ===")
print()
print(f"Total test transactions:     {len(y_test):,}")
print(f"Autonomously decided:        {autonomous.sum():,} ({autonomous.mean()*100:.1f}%)")
print(f"Deferred to human:           {deferred.sum():,}  ({deferred.mean()*100:.1f}%)")
print()

# Evaluate only autonomous decisions
y_test_np = y_test.values
autonomous_preds  = class_pred[autonomous]
autonomous_labels = y_test_np[autonomous]

print("=== AUTONOMOUS DECISIONS ONLY ===")
print(f"Accuracy: {accuracy_score(autonomous_labels, autonomous_preds):.4f}")
print(f"F1 Score: {f1_score(autonomous_labels, autonomous_preds):.4f}")
print()
print(classification_report(autonomous_labels, autonomous_preds,
      target_names=['Normal', 'Fraud']))

# How many frauds were deferred vs missed?
fraud_deferred  = deferred[y_test_np == 1].sum()
fraud_autonomous_wrong = ((class_pred[autonomous] != y_test_np[autonomous]) & 
                           (y_test_np[autonomous] == 1)).sum()

print(f"Frauds correctly deferred to human: {fraud_deferred}")
print(f"Frauds missed autonomously:         {fraud_autonomous_wrong}")

=== STANDARD L2D RESULTS ===

Total test transactions:     56,962
Autonomously decided:        56,962 (100.0%)
Deferred to human:           0  (0.0%)

=== AUTONOMOUS DECISIONS ONLY ===
Accuracy: 0.9989
F1 Score: 0.7249

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.63      0.85      0.72        98

    accuracy                           1.00     56962
   macro avg       0.82      0.92      0.86     56962
weighted avg       1.00      1.00      1.00     56962

Frauds correctly deferred to human: 0
Frauds missed autonomously:         15


In [15]:
# ── BLOCK 6B: Retrain with Lower Deferral Cost ──

# Reset model
model2 = L2DNetwork(input_dim=input_dim)
optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

# Try lower cost — makes deferral cheaper so model defers more
DEFERRAL_COST = 0.1
num_epochs = 10
train_losses2 = []

print(f"Retraining with deferral cost = {DEFERRAL_COST}")
print("=" * 45)

for epoch in range(num_epochs):
    model2.train()
    epoch_loss = 0
    num_batches = 0
    
    for X_batch, y_batch in train_loader:
        optimizer2.zero_grad()
        class_logits, defer_prob = model2(X_batch)
        loss = l2d_loss(class_logits, defer_prob, 
                        y_batch, cost=DEFERRAL_COST)
        loss.backward()
        optimizer2.step()
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    train_losses2.append(avg_loss)
    print(f"Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg_loss:.4f}")

print("=" * 45)
print("✅ Retraining complete!")

Retraining with deferral cost = 0.1
Epoch  1/10 | Loss: 0.0765
Epoch  2/10 | Loss: 0.0563
Epoch  3/10 | Loss: 0.0539
Epoch  4/10 | Loss: 0.0529
Epoch  5/10 | Loss: 0.0524
Epoch  6/10 | Loss: 0.0520
Epoch  7/10 | Loss: 0.0515
Epoch  8/10 | Loss: 0.0513
Epoch  9/10 | Loss: 0.0512
Epoch 10/10 | Loss: 0.0510
✅ Retraining complete!


In [16]:
# ── BLOCK 7: Evaluate Retrained L2D (cost=0.1) ──

model2.eval()
with torch.no_grad():
    class_logits, defer_prob = model2(X_test_tensor)
    
    class_pred = torch.argmax(class_logits, dim=1).numpy()
    defer_p    = defer_prob.squeeze().numpy()
    deferred   = defer_p > 0.5
    autonomous = ~deferred

print("=== STANDARD L2D RESULTS (cost=0.1) ===")
print()
print(f"Total test transactions:     {len(y_test):,}")
print(f"Autonomously decided:        {autonomous.sum():,} ({autonomous.mean()*100:.1f}%)")
print(f"Deferred to human:           {deferred.sum():,} ({deferred.mean()*100:.1f}%)")
print()

y_test_np = y_test.values

if autonomous.sum() > 0:
    autonomous_preds  = class_pred[autonomous]
    autonomous_labels = y_test_np[autonomous]
    
    print("=== AUTONOMOUS DECISIONS ONLY ===")
    print(f"Accuracy: {accuracy_score(autonomous_labels, autonomous_preds):.4f}")
    print(f"F1 Score: {f1_score(autonomous_labels, autonomous_preds):.4f}")
    print()
    print(classification_report(autonomous_labels, autonomous_preds,
          target_names=['Normal', 'Fraud']))
    
    fraud_deferred = deferred[y_test_np == 1].sum()
    fraud_missed   = ((class_pred[autonomous] != y_test_np[autonomous]) & 
                      (y_test_np[autonomous] == 1)).sum()
    
    print(f"Frauds deferred to human:  {fraud_deferred}")
    print(f"Frauds missed autonomously: {fraud_missed}")
else:
    print("Model deferred everything — cost too low, try 0.2")

=== STANDARD L2D RESULTS (cost=0.1) ===

Total test transactions:     56,962
Autonomously decided:        56,115 (98.5%)
Deferred to human:           847 (1.5%)

=== AUTONOMOUS DECISIONS ONLY ===
Accuracy: 0.9998
F1 Score: 0.0000

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56106
       Fraud       0.00      0.00      0.00         9

    accuracy                           1.00     56115
   macro avg       0.50      0.50      0.50     56115
weighted avg       1.00      1.00      1.00     56115

Frauds deferred to human:  89
Frauds missed autonomously: 9


In [17]:
# ── BLOCK 8: Find Best Deferral Cost ──

# Try multiple cost values to find the sweet spot
costs_to_try = [0.2, 0.3, 0.4]
results = []

for cost_val in costs_to_try:
    # Train fresh model
    m = L2DNetwork(input_dim=input_dim)
    opt = optim.Adam(m.parameters(), lr=0.001)
    
    print(f"Training with cost={cost_val}...")
    
    for epoch in range(10):
        m.train()
        for X_batch, y_batch in train_loader:
            opt.zero_grad()
            cl, dp = m(X_batch)
            loss = l2d_loss(cl, dp, y_batch, cost=cost_val)
            loss.backward()
            opt.step()
    
    # Evaluate
    m.eval()
    with torch.no_grad():
        cl, dp = m(X_test_tensor)
        cp = torch.argmax(cl, dim=1).numpy()
        dp = dp.squeeze().numpy()
        deferred = dp > 0.5
        autonomous = ~deferred
        y_np = y_test.values
        
        deferral_rate = deferred.mean() * 100
        fraud_deferred = deferred[y_np == 1].sum()
        
        if autonomous.sum() > 0:
            f1 = f1_score(y_np[autonomous], cp[autonomous], 
                         zero_division=0)
            fraud_missed = ((cp[autonomous] != y_np[autonomous]) & 
                           (y_np[autonomous] == 1)).sum()
        else:
            f1 = 0
            fraud_missed = 0
        
        results.append({
            'cost': cost_val,
            'deferral_rate': deferral_rate,
            'fraud_deferred': fraud_deferred,
            'fraud_missed': fraud_missed,
            'f1_autonomous': f1
        })
        
        print(f"  Cost={cost_val} | Deferral: {deferral_rate:.1f}% | "
              f"Frauds deferred: {fraud_deferred} | "
              f"Frauds missed: {fraud_missed} | "
              f"F1: {f1:.4f}")

print()
print("=== SUMMARY ===")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Training with cost=0.2...
  Cost=0.2 | Deferral: 0.0% | Frauds deferred: 0 | Frauds missed: 18 | F1: 0.7080
Training with cost=0.3...
  Cost=0.3 | Deferral: 0.0% | Frauds deferred: 0 | Frauds missed: 14 | F1: 0.7210
Training with cost=0.4...
  Cost=0.4 | Deferral: 0.0% | Frauds deferred: 0 | Frauds missed: 15 | F1: 0.7217

=== SUMMARY ===
 cost  deferral_rate  fraud_deferred  fraud_missed  f1_autonomous
  0.2       0.001756               0            18       0.707965
  0.3       0.000000               0            14       0.721030
  0.4       0.000000               0            15       0.721739


In [18]:
# ── BLOCK 9: Fine-tune between 0.1 and 0.2 ──

costs_to_try = [0.11, 0.13, 0.15, 0.17]
results2 = []

for cost_val in costs_to_try:
    m = L2DNetwork(input_dim=input_dim)
    opt = optim.Adam(m.parameters(), lr=0.001)
    
    print(f"Training with cost={cost_val}...")
    
    for epoch in range(10):
        m.train()
        for X_batch, y_batch in train_loader:
            opt.zero_grad()
            cl, dp = m(X_batch)
            loss = l2d_loss(cl, dp, y_batch, cost=cost_val)
            loss.backward()
            opt.step()
    
    m.eval()
    with torch.no_grad():
        cl, dp = m(X_test_tensor)
        cp = torch.argmax(cl, dim=1).numpy()
        dp = dp.squeeze().numpy()
        deferred = dp > 0.5
        autonomous = ~deferred
        y_np = y_test.values
        
        deferral_rate  = deferred.mean() * 100
        fraud_deferred = deferred[y_np == 1].sum()
        
        if autonomous.sum() > 0:
            f1 = f1_score(y_np[autonomous], cp[autonomous],
                         zero_division=0)
            fraud_missed = ((cp[autonomous] != y_np[autonomous]) &
                           (y_np[autonomous] == 1)).sum()
        else:
            f1 = 0
            fraud_missed = 0
        
        results2.append({
            'cost': cost_val,
            'deferral_rate': round(deferral_rate, 2),
            'fraud_deferred': int(fraud_deferred),
            'fraud_missed': int(fraud_missed),
            'f1_autonomous': round(f1, 4)
        })
        
        print(f"  Cost={cost_val} | Deferral: {deferral_rate:.1f}% | "
              f"Frauds deferred: {fraud_deferred} | "
              f"Frauds missed: {fraud_missed} | "
              f"F1: {f1:.4f}")

print()
print("=== FINE-TUNING SUMMARY ===")
print(pd.DataFrame(results2).to_string(index=False))

Training with cost=0.11...
  Cost=0.11 | Deferral: 0.6% | Frauds deferred: 1 | Frauds missed: 12 | F1: 0.7083
Training with cost=0.13...
  Cost=0.13 | Deferral: 0.6% | Frauds deferred: 7 | Frauds missed: 12 | F1: 0.7980
Training with cost=0.15...
  Cost=0.15 | Deferral: 0.4% | Frauds deferred: 0 | Frauds missed: 15 | F1: 0.6694
Training with cost=0.17...
  Cost=0.17 | Deferral: 0.3% | Frauds deferred: 0 | Frauds missed: 13 | F1: 0.7727

=== FINE-TUNING SUMMARY ===
 cost  deferral_rate  fraud_deferred  fraud_missed  f1_autonomous
 0.11           0.57               1            12         0.7083
 0.13           0.64               7            12         0.7980
 0.15           0.37               0            15         0.6694
 0.17           0.31               0            13         0.7727


In [11]:
# ── BLOCK 10: Final Standard L2D Model (cost=0.13) ──
# NOTE: torch.manual_seed added during verification pass (22-23 Jul 2026).
# Model weight init and batch shuffling were previously unseeded, so this
# specific run could not be reproduced on demand. See Master Reference
# Document Section 11 for the full explanation.

BEST_COST = 0.13

# Reproducibility
torch.manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,
                           generator=torch.Generator().manual_seed(42))

# Train final model
final_l2d = L2DNetwork(input_dim=input_dim)
optimizer_final = optim.Adam(final_l2d.parameters(), lr=0.001)

print(f"Training Final Standard L2D (cost={BEST_COST})...")
print("=" * 45)

final_losses = []
for epoch in range(15):  # 15 epochs for final model
    final_l2d.train()
    epoch_loss = 0
    num_batches = 0

    for X_batch, y_batch in train_loader:
        optimizer_final.zero_grad()
        cl, dp = final_l2d(X_batch)
        loss = l2d_loss(cl, dp, y_batch, cost=BEST_COST)
        loss.backward()
        optimizer_final.step()
        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    final_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0 or epoch == 14:
        print(f"Epoch {epoch+1:2d}/15 | Loss: {avg_loss:.4f}")

print("=" * 45)
print("✅ Final Standard L2D trained!")

Training Final Standard L2D (cost=0.13)...
Epoch  5/15 | Loss: 0.0153
Epoch 10/15 | Loss: 0.0075
Epoch 15/15 | Loss: 0.0050
✅ Final Standard L2D trained!

In [12]:
# ── BLOCK 11: Full Evaluation of Standard L2D ──

final_l2d.eval()
with torch.no_grad():
    cl, dp = final_l2d(X_test_tensor)
    cp     = torch.argmax(cl, dim=1).numpy()
    dp     = dp.squeeze().numpy()
    deferred   = dp > 0.5
    autonomous = ~deferred
    y_np   = y_test.values
    rt_np  = rt_test.values

# ── Overall stats ──
print("=" * 50)
print("     STANDARD L2D — FULL EVALUATION REPORT")
print("=" * 50)
print()
print(f"Total test transactions:       {len(y_np):,}")
print(f"Autonomously decided:          {autonomous.sum():,} ({autonomous.mean()*100:.2f}%)")
print(f"Deferred to human:             {deferred.sum():,} ({deferred.mean()*100:.2f}%)")
print()

# ── Autonomous performance ──
auto_preds  = cp[autonomous]
auto_labels = y_np[autonomous]

print("─" * 50)
print("AUTONOMOUS DECISIONS:")
print(f"  Accuracy:  {accuracy_score(auto_labels, auto_preds):.4f}")
print(f"  F1 Score:  {f1_score(auto_labels, auto_preds, zero_division=0):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(auto_labels, cl[autonomous].numpy()[:,1]):.4f}")
print()
print(classification_report(auto_labels, auto_preds,
      target_names=['Normal','Fraud'], zero_division=0))

# ── Fraud breakdown ──
print("─" * 50)
print("FRAUD BREAKDOWN:")
total_frauds   = (y_np == 1).sum()
fraud_deferred = deferred[y_np == 1].sum()
fraud_caught   = ((cp[autonomous] == 1) & (y_np[autonomous] == 1)).sum()
fraud_missed   = ((cp[autonomous] == 0) & (y_np[autonomous] == 1)).sum()

print(f"  Total frauds in test:        {total_frauds}")
print(f"  Frauds deferred to human:    {fraud_deferred}")
print(f"  Frauds caught autonomously:  {fraud_caught}")
print(f"  Frauds missed:               {fraud_missed}")
print()

# ── Risk tier deferral breakdown ──
print("─" * 50)
print("DEFERRAL BY RISK TIER:")
for tier in ['Low', 'Medium', 'High', 'Critical']:
    tier_mask     = rt_np == tier
    tier_deferred = deferred[tier_mask].sum()
    tier_total    = tier_mask.sum()
    tier_rate     = (tier_deferred / tier_total) * 100
    print(f"  {tier:10s}: {tier_deferred:4d} deferred "
          f"out of {tier_total:6d} ({tier_rate:.2f}%)")

print()
print("=" * 50)
print("✅ Evaluation complete!")

     STANDARD L2D ── FULL EVALUATION REPORT

Total test transactions:       56,962
Autonomously decided:          56,822 (99.75%)
Deferred to human:             140 (0.25%)

──────────────────────────────────────────────────
AUTONOMOUS DECISIONS:
  Accuracy:  0.9992
  F1 Score:  0.7736
  ROC-AUC:   0.9787

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56724
       Fraud       0.72      0.85      0.77        98

    accuracy                           1.00     56822
   macro avg       0.86      0.92      0.89     56822
weighted avg       1.00      1.00      1.00     56822

──────────────────────────────────────────────────
FRAUD BREAKDOWN:
  Total frauds in test:        98
  Frauds deferred to human:    1
  Frauds caught autonomously:  82
  Frauds missed:               15

──────────────────────────────────────────────────
DEFERRAL BY RISK TIER:
  Low       : 0.28% deferred
  Medium    : 0.18% deferred
  High      : 0.25% deferred


In [13]:
# ── BLOCK 12: Save Standard L2D Results ──
import json
import os

# Save model
os.makedirs(r"E:\NCI\Sem3\Thesis\implementation\models", exist_ok=True)
torch.save(final_l2d.state_dict(),
           r"E:\NCI\Sem3\Thesis\implementation\models\standard_l2d.pth")

# Save results (seeded, reproducible -- seed=42)
l2d_results = {
    'model': 'Standard L2D',
    'deferral_cost': 0.13,
    'total_test': 56962,
    'autonomous': 56822,
    'deferred': 140,
    'deferral_rate': 0.25,
    'accuracy': 0.9992,
    'f1_score': 0.7736,
    'roc_auc': 0.9787,
    'frauds_total': 98,
    'frauds_deferred': 1,
    'frauds_caught': 82,
    'frauds_missed': 15,
    'deferral_by_tier': {
        'Low': 0.28,
        'Medium': 0.18,
        'High': 0.25,
        'Critical': 0.21
    },
    'seed': 42
}

with open(r"E:\NCI\Sem3\Thesis\implementation\results\standard_l2d_results.json", 'w') as f:
    json.dump(l2d_results, f, indent=4)

print("✅ Model and results saved!")
print()
print("=== COMPARISON SO FAR ===")
print()
print(f"{'Metric':<25} {'Baseline':>12} {'Standard L2D':>14}")
print("-" * 53)
print(f"{'F1 Score':<25} {'0.1094':>12} {'0.7736':>14}")
print(f"{'ROC-AUC':<25} {'0.9698':>12} {'0.9787':>14}")
print(f"{'Deferral Rate':<25} {'0.00%':>12} {'0.25%':>14}")
print(f"{'Frauds Missed':<25} {'8':>12} {'15':>14}")
print()
print("Next: Build Model 3 ─ Risk-Sensitive L2D")
print("Expected: Critical tier deferral >> Low tier deferral")

✅ Model and results saved!

=== COMPARISON SO FAR ===

Metric                        Baseline   Standard L2D
-----------------------------------------------------
F1 Score                        0.1094         0.7736
ROC-AUC                         0.9698         0.9787
Deferral Rate                    0.00%          0.25%
Frauds Missed                        8             15

Next: Build Model 3 ─ Risk-Sensitive L2D
Expected: Critical tier deferral >> Low tier deferral